In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

In [2]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

In [3]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu130
CUDA available: True


In [4]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [5]:
from google.colab import files

uploaded = files.upload()

Saving converted_data.csv to converted_data.csv


In [6]:
df = pd.read_csv("converted_data.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (34469, 3)

Columns:
['Sinhala', 'English', 'Singlish']


In [7]:
# Remove only "[Unkown]" while preserving the remaining words.
pattern = r"\[\s*Unkown\s*\]"

for column in ["English", "Singlish"]:
    df[column] = (
        df[column]
        .astype("string")
        .str.replace(pattern, "", case=False, regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print(f"Cleaning completed. All {len(df)} rows retained.")
display(df.head())

Cleaning completed. All 34469 rows retained.


,Sinhala,English,Singlish
0,c esc ma tete මෙය මගේ ප්‍රධාන අයිතියයි ඔබ ප්‍ර...,c esc ma tete this is my headright see you re ...,meya mage pradhana ayithiyayi oba prashnawaliy...
1,නෑ නෑ ඒක මගේ වරදක් අපිට හරි හමන් කැමරොන් කෙනෙක...,no no it s my fault we didn t have a proper in...,ne ne eka mage waradak apita hari haman kenek ...
2,හැක් කිරීම සහ කට වහගෙන කෙළ ගසන කොටස නොවේ කරුණා...,not the hacking and gagging and spitting part ...,hek kireema saha kata wahagena kela gasana kot...
3,"හොඳයි, මම හිතුවා අපි උච්චාරණයෙන් පටන් ගනිමු එය...",well i thought we d start with pronunciation i...,hondayi mama hithuwa api uchcharanayen patan g...
4,ඔයා මගෙන් අහනවා ඒක හරිම ලස්සනයි ඔයාගේ නම මොකක්...,you re asking me out that s so cute what s you...,oya magen ahanawa eka harima lassanayi oyage n...


In [8]:
# Keep only the translation columns
data = df[["English", "Singlish"]].copy()

In [9]:
# Rename columns for clarity
data = data.rename(columns={
    "English": "source_text",
    "Singlish": "target_text"
})

In [10]:
# Remove missing values
data = data.dropna(subset=["source_text", "target_text"])

In [11]:
# Convert to string
data["source_text"] = data["source_text"].astype(str).str.strip()
data["target_text"] = data["target_text"].astype(str).str.strip()

In [12]:
# Remove empty records
data = data[
    (data["source_text"] != "") &
    (data["target_text"] != "")
]

In [13]:
# Remove exact duplicate translation pairs
data = data.drop_duplicates(
    subset=["source_text", "target_text"]
).reset_index(drop=True)

In [14]:
print("Clean dataset size:", len(data))
display(data.head(10))

Clean dataset size: 34346


,source_text,target_text
0,c esc ma tete this is my headright see you re ...,meya mage pradhana ayithiyayi oba prashnawaliy...
1,no no it s my fault we didn t have a proper in...,ne ne eka mage waradak apita hari haman kenek ...
2,not the hacking and gagging and spitting part ...,hek kireema saha kata wahagena kela gasana kot...
3,well i thought we d start with pronunciation i...,hondayi mama hithuwa api uchcharanayen patan g...
4,you re asking me out that s so cute what s you...,oya magen ahanawa eka harima lassanayi oyage n...
5,gosh if only we could find kat a boyfriend let...,apita pemwathek hoyaganna puluwan nam mata kar...
6,that s because it s such a nice one forget french,e pransha amathaka karanna tharam lassana ekak...
7,how is our little find the wench a date plan p...,ape punchi soyageneema kohomada dina selesma h...
8,sure have i really really really wanna go but ...,mata eththatama eththatama yanna awashyada nam...
9,you have my word as a gentlemanyou re sweet,mahathmayeku lesa oba mage wachanaya oba mihir...


In [15]:
# print(data.isnull().sum())
# print("Duplicates:", data.duplicated().sum())

In [16]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, temp_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 27476
Validation: 3435
Testing: 3435


In [17]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['source_text', 'target_text'],
    num_rows: 27476
})
Dataset({
    features: ['source_text', 'target_text'],
    num_rows: 3435
})
Dataset({
    features: ['source_text', 'target_text'],
    num_rows: 3435
})


In [18]:
MODEL_NAME = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Model loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/mt5-small


In [19]:
MAX_SOURCE_LENGTH = 64
MAX_TARGET_LENGTH = 64

def preprocess_function(examples):
    inputs = [
        f"translate English to Singlish: {text}"
        for text in examples["source_text"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [20]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

print(tokenized_train)

Map:   0%|          | 0/27476 [00:00<?, ? examples/s]

Map:   0%|          | 0/3435 [00:00<?, ? examples/s]

Map:   0%|          | 0/3435 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 27476
})


In [21]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [22]:
OUTPUT_DIR = "./mt5-english-singlish"


training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=2,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=3e-4,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    predict_with_generate=True,

    logging_strategy="steps",
    logging_steps=10,

    save_total_limit=2,

    report_to="none",

    # Use full precision for this retraining experiment.
    fp16=False,
    bf16=False,
    optim="adafactor",
    warmup_steps=39,
    gradient_accumulation_steps=2,
)

In [23]:
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

In [24]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.18.0


In [25]:
import inspect
print(inspect.signature(Seq2SeqTrainer.__init__))

(self, model: Union[ForwardRef('PreTrainedModel'), torch.nn.modules.module.Module, NoneType] = None, args: Optional[ForwardRef('TrainingArguments')] = None, data_collator: Optional[ForwardRef('DataCollator')] = None, train_dataset: Union[torch.utils.data.dataset.Dataset, ForwardRef('IterableDataset'), ForwardRef('datasets.Dataset'), NoneType] = None, eval_dataset: torch.utils.data.dataset.Dataset | dict[str, torch.utils.data.dataset.Dataset] | None = None, processing_class: Union[ForwardRef('PreTrainedTokenizerBase'), ForwardRef('BaseImageProcessor'), ForwardRef('FeatureExtractionMixin'), ForwardRef('ProcessorMixin'), NoneType] = None, model_init: collections.abc.Callable[[], 'PreTrainedModel'] | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[['EvalPrediction'], dict] | None = None, callbacks: list['TrainerCallback'] | None = None, optimizers: tuple[torch.optim.optimizer.Optimizer | None, torch.optim.lr_scheduler.Lambda

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,7.045102,3.060832
2,6.044985,2.736432


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=6870, training_loss=7.4425871520063245, metrics={'train_runtime': 3203.4559, 'train_samples_per_second': 17.154, 'train_steps_per_second': 2.145, 'total_flos': 2178568521277440.0, 'train_loss': 7.4425871520063245, 'epoch': 2.0})

In [27]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_test
)

print(test_results)

Training Loss,Validation Loss,Epoch
6.044985,2.711975,2


{'eval_loss': 2.711974620819092}


In [28]:
def translate_english_to_singlish(text):
    prompt = f"translate English to Singlish: {text}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4
        )

    translation = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    raw_text = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    if "<extra_id_" in raw_text or not translation.strip():
        raise RuntimeError(f"Failed translation for {text!r}: {raw_text!r}")
    return translation

In [29]:
examples = [
    "she okay i hope so",
    "they do to they do not",
    "that s because it s such a nice one forget french",
    "you have my word as a gentlemanyou re sweet",
    "well no then that s all you had to say"
]

for text in examples:
    print("English :", text)
    print("Singlish:", translate_english_to_singlish(text))
    print("-" * 50)

English : she okay i hope so
Singlish: eya hari mama hithanne ithin
--------------------------------------------------
English : they do to they do not
Singlish: owun karanne nehe
--------------------------------------------------
English : that s because it s such a nice one forget french
Singlish: eka hondayi eka hondayi eka hondayi
--------------------------------------------------
English : you have my word as a gentlemanyou re sweet
Singlish: obata mage prashnayak awashyayi obata awashyayi
--------------------------------------------------
English : well no then that s all you had to say
Singlish: hondayi ne ehenam e oyata kiyanna
--------------------------------------------------


In [30]:
# Block export of checkpoints that still generate pretraining placeholders.
MODEL_PASSED_SANITY_CHECK = False
validation_examples = []
for row in val_df.itertuples(index=False):
    prediction = translate_english_to_singlish(row.source_text)
    validation_examples.append({
        "English": row.source_text,
        "Reference": row.target_text,
        "Prediction": prediction,
    })
display(pd.DataFrame(validation_examples))
MODEL_PASSED_SANITY_CHECK = True
print("Placeholder sanity check passed. Review translation meanings before deployment.")


,English,Reference,Prediction
0,this is crazy that s why it will be so much fun,meka pissuwak eka nisa godak winoda weyi,meya pissu eya itha hondayi eya itha hondayi
1,i am sorry madame it is not permitted do you k...,mata samawenna medam ekata awasara ne oya dann...,mata samawenna mata samawenna me mage thaththa...
2,could you do somethin for me anything,oba ma wenuwen onema deyak karanawada,obata mata kisima deyak karanna puluwan kiyala
3,relax you got one fifty you heard the guy rick...,wiweka ganna obata eka panahak lebuna obata eh...,samahara wita obata awashyayi obata awashyayi ...
4,then we ll die my soul is prepared how s yours,ewita api meremu mage athmaya obe keseda yanna...,ewita api mage jeewithaya karana bawayi
...,...,...,...
3430,why haven t you called why don t you answer yo...,eyi oya katha kale neththe eyi oya oyage fon e...,eyi oya katha karanne neththe eyi mama danne n...
3431,your guest happens to be six foot five bright ...,obe aganthukaya adi hayak pahak deepthimath ra...,obe awurudu awurudu awurudu awurudu awurudu aw...
3432,is there a crowd in the street no sir,pare senagak innawada sar,ehi awurudu ekak ne sar
3433,korda escaped and you think he ll,penala giya oya hithanne eya weyi kiyala,oba sithanne ohu sithanne ohu sithanne ohu sit...


Placeholder sanity check passed. Review translation meanings before deployment.


In [31]:
assert MODEL_PASSED_SANITY_CHECK, "Run and pass the validation cell before saving/uploading."

SAVE_PATH = "./english-singlish-mt5"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./english-singlish-mt5


In [32]:
assert MODEL_PASSED_SANITY_CHECK, "Run and pass the validation cell before saving/uploading."

from google.colab import files
import shutil
import os

# Create a final project folder
final_folder = "/content/final_training_output"
os.makedirs(final_folder, exist_ok=True)

# Save final dataset
data.to_csv(
    f"{final_folder}/final_english_singlish_translation_dataset.csv",
    index=False
)

# Save trained model
trainer.save_model(f"{final_folder}/english-singlish-mt5")
tokenizer.save_pretrained(f"{final_folder}/english-singlish-mt5")

# Compress everything
zip_path = shutil.make_archive(
    "/content/english-singlish-training-final",
    "zip",
    final_folder
)

# Download ZIP
files.download(zip_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
!pip install -q -U huggingface_hub

In [35]:
from huggingface_hub import login

login()

In [36]:
assert MODEL_PASSED_SANITY_CHECK, "Run and pass the validation cell before saving/uploading."

from huggingface_hub import HfApi

api = HfApi()

repo_id = "Madushanavod/english-singlish-mt5"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

api.upload_folder(
    folder_path="/content/english-singlish-mt5",
    repo_id=repo_id,
    repo_type="model"
)

print("Model uploaded successfully!")

Model uploaded successfully!
